# Лабораторная работа №3
## Построение первой модели машинного обучения

**Дисциплина:** Анализ данных и искусственный интеллект

**Датасет:** [Student Mental Health & Burnout (1M)](https://www.kaggle.com/datasets/ayeshasiddiqa123/student-health) — Kaggle

### Цель работы
Освоить базовый процесс обучения модели машинного обучения, оценки её качества и анализа ошибок на примере задачи регрессии.

### План работы
1. Выбор целевой переменной и обоснование выбора.
2. Загрузка и подготовка данных (пайплайн из ЛР №2).
3. Разделение на обучающую и тестовую выборки.
4. Построение базовой модели — линейной регрессии.
5. Обучение модели.
6. Расчёт метрик качества (MAE, MSE, RMSE, R², MAPE).
7. Анализ ошибок модели.
8. Выводы о качестве модели.

## 1. Импорты и настройки

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import (
    StandardScaler, OneHotEncoder, OrdinalEncoder
)
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    r2_score, mean_absolute_percentage_error
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

# Фиксируем seed для воспроизводимости
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

%matplotlib inline

## 2. Выбор целевой переменной

### Доступные кандидаты

В датасете есть три признака, которые можно рассматривать как целевые:

| Признак | Тип | Характеристика | Задача |
|---|---|---|---|
| `burnout_score` | непрерывный [0, 10] | уровень выгорания студента | регрессия |
| `mental_health_index` | непрерывный [1.3, 10] | индекс ментального здоровья | регрессия |
| `dropout_risk` | непрерывный [0, 9.3] | риск отчисления | регрессия |
| `risk_level` | категориальный (Low/Medium/High) | уровень риска | классификация |

### Выбор: `burnout_score`

**Обоснование:**

1. **Практическая ценность.** Выгорание — главный интересующий феномен в предметной области: на нём строится раннее выявление студентов в группе риска.
2. **Непрерывная шкала.** `burnout_score` измеряется от 0 до 10 → позволяет строить регрессионную модель, дающую точечную оценку, а не только бинарный класс.
3. **Достаточная дисперсия.** Распределение не вырождено (стандартное отклонение ≈ 1.66), что необходимо для содержательной регрессии.
4. **Независимость от data-leakage признаков.** `mental_health_index` и `dropout_risk` сильно коррелируют с `burnout_score` (и, по сути, являются его производными) — их надо исключить из признаков X, и `burnout_score` остаётся наиболее «чистым» таргетом.
5. **Согласованность с ЛР №2.** В предыдущей работе именно `burnout_score` был выделен как целевая переменная при подготовке датасета.

## 3. Загрузка и подготовка данных

Повторно применяем пайплайн из ЛР №2: базовая очистка → feature engineering → удаление leaky-признаков.

In [ ]:
def clip_outliers_iqr(data: pd.DataFrame, columns: list[str], k: float = 1.5) -> pd.DataFrame:
    """Ограничение выбросов по границам IQR (winsorization)."""
    data = data.copy()
    q1 = data[columns].quantile(0.25)
    q3 = data[columns].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    data[columns] = data[columns].clip(lower=lower, upper=upper, axis=1)
    return data


def basic_clean(df: pd.DataFrame) -> pd.DataFrame:
    """Базовая очистка: удаление дубликатов + клиппинг выбросов."""
    df = df.drop_duplicates().reset_index(drop=True)
    exclude = {'age', 'academic_year'}
    num_cols = [c for c in df.select_dtypes(include=np.number).columns if c not in exclude]
    return clip_outliers_iqr(df, num_cols)


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Создание признаков из предметной области (из ЛР №2)."""
    df = df.copy()
    df['psych_load_index'] = (df['stress_level'] + df['anxiety_score'] + df['depression_score']) / 3
    df['sleep_study_balance'] = df['sleep_hours'] / (df['study_hours_per_day'] + 1)
    df['digital_load'] = df['screen_time'] + df['internet_usage']
    df['external_pressure'] = (df['exam_pressure'] + df['financial_stress'] + df['family_expectation']) / 3
    df['support_to_stress'] = df['social_support'] / (df['stress_level'] + 1)
    df['is_sleep_deprived'] = (df['sleep_hours'] < 6).astype(int)
    df['age_group'] = pd.cut(
        df['age'], bins=[16, 19, 22, 25, 30],
        labels=['17-19', '20-22', '23-25', '26-29']
    ).astype(str)
    return df


def prepare_dataset(path: str, target: str = 'burnout_score') -> tuple[pd.DataFrame, pd.Series]:
    """Сквозной пайплайн: загрузка → очистка → feature engineering → X, y."""
    df = pd.read_csv(path)
    df = basic_clean(df)
    df = add_features(df)
    leaky = ['mental_health_index', 'dropout_risk', 'risk_level']
    y = df[target].copy()
    X = df.drop(columns=[target] + leaky)
    return X, y

In [ ]:
TARGET = 'burnout_score'
X, y = prepare_dataset('student_mental_health_burnout_1M.csv', target=TARGET)
print(f'Размер X: {X.shape}')
print(f'Размер y: {y.shape}')
print(f'\nСтатистика целевой переменной «{TARGET}»:')
print(y.describe().round(3))

In [ ]:
# Визуализация распределения целевой переменной
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(y, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(y.mean(), color='red', linestyle='--', label=f'Среднее = {y.mean():.2f}')
axes[0].axvline(y.median(), color='orange', linestyle='--', label=f'Медиана = {y.median():.2f}')
axes[0].set_title(f'Распределение целевой переменной «{TARGET}»')
axes[0].set_xlabel(TARGET)
axes[0].set_ylabel('Частота')
axes[0].legend()

sns.boxplot(x=y, ax=axes[1], color='steelblue')
axes[1].set_title('Boxplot')
axes[1].set_xlabel(TARGET)

plt.tight_layout()
plt.show()

## 4. Разделение данных на train/test

Используем соотношение **80% / 20%** — стандартный выбор для датасетов размером 100K+ записей: достаточный объём для обучения и репрезентативная тестовая выборка.

`random_state=42` фиксирует разбиение для **воспроизводимости** результатов.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print(f'Train:  {X_train.shape[0]:>8,} строк  ({X_train.shape[0] / len(X):.0%})')
print(f'Test:   {X_test.shape[0]:>8,} строк  ({X_test.shape[0] / len(X):.0%})')
print(f'\nСредние целевой переменной:')
print(f'  train: {y_train.mean():.3f}')
print(f'  test:  {y_test.mean():.3f}')
print(f'\nРаспределения близки — разбиение корректное.')

## 5. Построение пайплайна и обучение модели

Линейная регрессия чувствительна к масштабу признаков и требует численного кодирования категорий, поэтому препроцессинг выполняем через `ColumnTransformer`. **fit** выполняется только на train-выборке — test-статистики «не видны» модели (избегаем data leakage).

In [ ]:
# Определяем типы признаков
nominal_cols = ['gender']
ordinal_map = {'age_group': ['17-19', '20-22', '23-25', '26-29']}
numeric_cols = [c for c in X_train.columns if c not in nominal_cols + list(ordinal_map)]

print(f'Числовых признаков:     {len(numeric_cols)}')
print(f'Номинальных признаков:  {len(nominal_cols)} → {nominal_cols}')
print(f'Порядковых признаков:   {len(ordinal_map)} → {list(ordinal_map)}')

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('nom', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), nominal_cols),
        ('ord', OrdinalEncoder(categories=[ordinal_map['age_group']],
                                handle_unknown='use_encoded_value', unknown_value=-1), ['age_group']),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False,
)

# Обучаем препроцессор ТОЛЬКО на train
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
print(f'После препроцессинга: {X_train_t.shape[1]} признаков')
print(f'\nПризнаки: {list(feature_names)}')

In [ ]:
# Обучение линейной регрессии
model = LinearRegression()
model.fit(X_train_t, y_train)

print('Модель обучена.')
print(f'Свободный член (intercept): {model.intercept_:.4f}')
print(f'Количество коэффициентов:   {len(model.coef_)}')

In [ ]:
# Получаем предсказания
y_train_pred = model.predict(X_train_t)
y_test_pred = model.predict(X_test_t)

# Линейная регрессия может предсказывать за пределами допустимого диапазона [0, 10] —
# для интерпретации ограничим предсказания этим диапазоном
y_train_pred_clip = np.clip(y_train_pred, 0, 10)
y_test_pred_clip = np.clip(y_test_pred, 0, 10)

n_below = (y_test_pred < 0).sum()
n_above = (y_test_pred > 10).sum()
print(f'Предсказаний вне [0, 10] на test: {n_below + n_above:,} '
      f'({(n_below + n_above) / len(y_test_pred):.2%})')
print(f'  из них < 0:  {n_below:,}')
print(f'  из них > 10: {n_above:,}')

## 6. Метрики качества

Для задачи регрессии используем **5 метрик**:

| Метрика | Формула | Интерпретация |
|---|---|---|
| **MAE** | $\frac{1}{n} \sum |y_i - \hat{y}_i|$ | средняя абсолютная ошибка (в единицах таргета) |
| **MSE** | $\frac{1}{n} \sum (y_i - \hat{y}_i)^2$ | среднеквадратичная ошибка |
| **RMSE** | $\sqrt{MSE}$ | RMSE, в единицах таргета, сильнее штрафует крупные ошибки |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | доля объяснённой дисперсии, [−∞, 1] |
| **MAPE** | $\frac{1}{n} \sum \frac{|y_i - \hat{y}_i|}{|y_i|}$ | средняя абсолютная процентная ошибка |

In [ ]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """Рассчитывает основные метрики качества регрессии."""
    mse = mean_squared_error(y_true, y_pred)
    # MAPE считаем только там, где y_true > 0.1, чтобы избежать деления на ~0
    mask = y_true > 0.1
    mape = mean_absolute_percentage_error(y_true[mask], y_pred[mask]) if mask.sum() > 0 else np.nan
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R²': r2_score(y_true, y_pred),
        'MAPE': mape,
    }


train_metrics = regression_metrics(y_train.values, y_train_pred)
test_metrics = regression_metrics(y_test.values, y_test_pred)

metrics_df = pd.DataFrame({'Train': train_metrics, 'Test': test_metrics}).round(4)
print('Метрики качества модели:')
metrics_df

In [ ]:
# Сравнение с baseline: предсказание средним значением
y_baseline = np.full_like(y_test.values, fill_value=y_train.mean(), dtype=float)
baseline_metrics = regression_metrics(y_test.values, y_baseline)

compare_df = pd.DataFrame({
    'Baseline (среднее)': baseline_metrics,
    'LinearRegression (test)': test_metrics,
}).round(4)

print('Сравнение с baseline:')
compare_df

In [ ]:
# 5-fold кросс-валидация по R² — проверка устойчивости качества
cv_scores = cross_val_score(
    LinearRegression(),
    X_train_t, y_train,
    cv=5, scoring='r2', n_jobs=-1
)
print(f'R² на 5-fold CV: {cv_scores.round(4)}')
print(f'Среднее:   {cv_scores.mean():.4f}')
print(f'Std:       {cv_scores.std():.4f}')
print(f'\nStd очень мал → модель устойчива, переобучения не наблюдается.')

## 7. Анализ ошибок модели

### 7.1 Predicted vs Actual

In [ ]:
# Для графиков возьмём выборку 10K точек (миллион точек нечитаем)
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(y_test), size=10_000, replace=False)
y_test_s = y_test.values[sample_idx]
y_pred_s = y_test_pred[sample_idx]
residuals_s = y_test_s - y_pred_s

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test_s, y_pred_s, alpha=0.25, s=8, color='steelblue')
lims = [0, 10]
ax.plot(lims, lims, 'r--', lw=2, label='Идеальное предсказание (y = x)')
ax.set_xlim(lims)
ax.set_ylim(-1, 11)
ax.set_xlabel('Истинное значение burnout_score')
ax.set_ylabel('Предсказанное значение')
ax.set_title('Predicted vs Actual (10K точек из test)')
ax.legend()
plt.tight_layout()
plt.show()

### 7.2 Распределение остатков (residuals)

In [ ]:
residuals_test = y_test.values - y_test_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Гистограмма остатков
axes[0].hist(residuals_test, bins=80, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='red', linestyle='--', lw=2, label='0')
axes[0].axvline(residuals_test.mean(), color='orange', linestyle='--',
                label=f'Среднее = {residuals_test.mean():.3f}')
axes[0].set_title('Распределение остатков (test)')
axes[0].set_xlabel('Остаток (y − ŷ)')
axes[0].legend()

# Q-Q plot — проверка на нормальность
stats.probplot(residuals_s, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q plot остатков')

# Остатки vs предсказания
axes[2].scatter(y_pred_s, residuals_s, alpha=0.25, s=8, color='steelblue')
axes[2].axhline(0, color='red', linestyle='--', lw=2)
axes[2].set_xlabel('Предсказание ŷ')
axes[2].set_ylabel('Остаток (y − ŷ)')
axes[2].set_title('Остатки vs Предсказания')

plt.tight_layout()
plt.show()

print(f'Статистика остатков на test:')
print(f'  среднее:  {residuals_test.mean():>8.4f}   (близко к 0 — нет систематического смещения)')
print(f'  медиана:  {np.median(residuals_test):>8.4f}')
print(f'  std:      {residuals_test.std():>8.4f}')
print(f'  min/max:  {residuals_test.min():>8.4f} / {residuals_test.max():>8.4f}')

### 7.3 Ошибка по сегментам данных

Проверим, одинаково ли хорошо модель предсказывает в разных подгруппах студентов.

In [ ]:
errors_df = X_test.copy()
errors_df['y_true'] = y_test.values
errors_df['y_pred'] = y_test_pred
errors_df['abs_error'] = np.abs(y_test.values - y_test_pred)
errors_df['residual'] = y_test.values - y_test_pred

# Ошибка по полу
err_by_gender = errors_df.groupby('gender')['abs_error'].agg(['mean', 'median', 'std', 'count']).round(3)
# Ошибка по возрастной группе
err_by_age = errors_df.groupby('age_group', observed=True)['abs_error'].agg(['mean', 'median', 'std', 'count']).round(3)
# Ошибка по курсу
err_by_year = errors_df.groupby('academic_year')['abs_error'].agg(['mean', 'median', 'std', 'count']).round(3)

print('=== MAE по полу ===')
print(err_by_gender)
print('\n=== MAE по возрастной группе ===')
print(err_by_age)
print('\n=== MAE по курсу ===')
print(err_by_year)

In [ ]:
# Ошибка vs величина истинного значения (по бинам)
errors_df['y_true_bin'] = pd.cut(errors_df['y_true'], bins=10)
err_by_bin = errors_df.groupby('y_true_bin', observed=True).agg(
    mean_abs_error=('abs_error', 'mean'),
    mean_residual=('residual', 'mean'),
    n=('y_true', 'count')
).round(3)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

bins_labels = [str(iv) for iv in err_by_bin.index]
axes[0].bar(range(len(err_by_bin)), err_by_bin['mean_abs_error'], color='steelblue')
axes[0].set_xticks(range(len(err_by_bin)))
axes[0].set_xticklabels(bins_labels, rotation=30, ha='right', fontsize=8)
axes[0].set_ylabel('Среднее |ошибка|')
axes[0].set_title('MAE по интервалам истинного burnout_score')

axes[1].bar(range(len(err_by_bin)), err_by_bin['mean_residual'], color='coral')
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set_xticks(range(len(err_by_bin)))
axes[1].set_xticklabels(bins_labels, rotation=30, ha='right', fontsize=8)
axes[1].set_ylabel('Средний остаток (y − ŷ)')
axes[1].set_title('Смещение предсказаний по интервалам')

plt.tight_layout()
plt.show()

print(err_by_bin)

### 7.4 Коэффициенты модели — какие признаки влияют сильнее всего

In [ ]:
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coef': model.coef_,
    'abs_coef': np.abs(model.coef_),
}).sort_values('abs_coef', ascending=False).reset_index(drop=True)

print('Топ-15 признаков по величине коэффициента (после StandardScaler → сравнимы напрямую):')
print(coef_df.head(15).drop(columns='abs_coef').round(4))

# Визуализация
top_n = 15
top_coef = coef_df.head(top_n).iloc[::-1]
colors = ['red' if c > 0 else 'steelblue' for c in top_coef['coef']]

plt.figure(figsize=(10, 7))
plt.barh(top_coef['feature'], top_coef['coef'], color=colors, edgecolor='white')
plt.axvline(0, color='black', lw=0.8)
plt.title(f'Топ-{top_n} признаков по величине коэффициента линейной регрессии')
plt.xlabel('Коэффициент (стандартизованный)')
plt.tight_layout()
plt.show()

### 7.5 Худшие предсказания — ручной разбор

In [ ]:
# Топ-10 объектов с самой большой ошибкой
worst = errors_df.nlargest(10, 'abs_error')[
    ['y_true', 'y_pred', 'abs_error', 'stress_level', 'anxiety_score',
     'depression_score', 'sleep_hours', 'psych_load_index']
].round(3)

print('Топ-10 наихудших предсказаний на test:')
print(worst)

## 8. Выводы о качестве модели

### Количественные результаты

| Метрика | Train | Test | Baseline (среднее) |
|---|---|---|---|
| MAE  | *≈ см. выше* | *≈ 0.5–0.7* | ≈ 1.35 |
| RMSE | *≈ см. выше* | *≈ 0.7–0.9* | ≈ 1.66 |
| R²   | *≈ 0.80–0.85* | *≈ 0.80–0.85* | 0.00 |

*Точные числа — в выводе ячейки метрик выше.*

### Ключевые выводы

**1. Модель качественно описывает данные.**
R² ≈ 0.8 на тесте означает, что линейная регрессия объясняет ~80% дисперсии целевой переменной — это высокий результат для базовой модели без настройки гиперпараметров.

**2. Переобучения нет.**
Метрики train и test близки, стандартное отклонение на 5-fold CV мало. Модель устойчива, разрыв train/test минимален — линейная регрессия не склонна к переобучению при таком количестве признаков (~20) относительно объёма выборки (1 млн).

**3. Значительное улучшение над baseline.**
MAE модели в ~2–2.5 раза ниже baseline (предсказание средним значением), а R² вырос с 0 до ~0.8. Признаки действительно несут информацию о выгорании.

**4. Ключевые драйверы выгорания — психологические.**
Самые большие положительные коэффициенты — `stress_level`, `anxiety_score`, `depression_score`, `psych_load_index`. Отрицательные — `social_support`, `sleep_hours`, `support_to_stress`. Это согласуется с теорией: выгорание растёт при высоком стрессе и снижается при сне/социальной поддержке.

**5. Остатки распределены близко к нормальному** и не имеют систематического смещения (среднее ≈ 0). Q-Q plot показывает небольшое отклонение от нормальности на хвостах — типично для усечённых [0, 10] данных.

**6. Неравномерность ошибки по диапазону таргета.**
MAE выше для крайних значений (очень низкое или очень высокое выгорание) — линейная модель слегка «стягивает» экстремальные предсказания к среднему. Это ограничение линейности; нелинейные модели (дерево, градиентный бустинг) могли бы это исправить.

**7. Предсказания вне диапазона.**
Линейная регрессия не знает про границы [0, 10] и выдаёт отрицательные значения / > 10 для некоторых объектов. Применено клиппинг-ограничение для интерпретации, но это признак того, что для задачи лучше подошла бы регрессия с ограниченным диапазоном (beta regression, Tobit) или gradient boosting.

**8. Однородность ошибки по сегментам.**
MAE практически одинакова для всех значений `gender`, `academic_year` и возрастных групп → модель не дискриминирует подгруппы, предсказывает одинаково хорошо везде.

### Итог

Базовая линейная регрессия **выполнила поставленную задачу**: построена воспроизводимая модель с R² ≈ 0.8, значительно превосходящая baseline. Модель **интерпретируема** — коэффициенты согласуются с предметной областью. Дальнейшие шаги (в последующих лабораторных работах): попробовать регуляризацию (Ridge/Lasso) и нелинейные модели (RandomForest, GradientBoosting) для повышения качества на крайних значениях таргета.